### Initialize the environement

In [1]:
from envs import my_matc_env_rb_strong

RENDER_MODE = 'sim'

# Initializing the environment
flight_route_file = 'routes/matc/auto_airspace_config.json'
aircraft_config_file = 'routes/matc/aircraft_configurations_auto.json'
env = my_matc_env_rb_strong.MATCEnv(flight_route_file, aircraft_config_file, render_mode=RENDER_MODE)

Using Python-based geo functions
Successfully loaded OpenAP performance model
Failed to load BADA performance model
Successfully loaded legacy performance model
Reading config from /Users/imansharifi/Documents/VirtualEnvs/ReinforcementLearning/Code/ATC_LLM/settings.cfg
Reading magnetic variation data
Loading global navigation database...
Reading cache: /Users/imansharifi/Documents/VirtualEnvs/ReinforcementLearning/Code/ATC_LLM/cache/navdata.p
Successfully loaded plugin AREA
Successfully loaded plugin DATAFEED


## Convert the raw ownship-intruder information to the prompts with answers

In [15]:
def get_ownship_prompt_v1(env, ownship_intruder_info):
    """
    Returns a formatted string prompt for the ownship information.
    """
    ownship_info = ownship_intruder_info['ownship_info']
    first_intruder_info = ownship_intruder_info['first_closest_front_intruder_info']
    second_intruder_info = ownship_intruder_info['second_closest_front_intruder_info']
    ownship_action = ownship_intruder_info['ownship_action']

    prompt = "You are an airspace tactical deconfliction assistant. " + \
            "Your task is to help the ownship aircraft avoid collisions with front intruder aircraft by suggesting appropriate speed adjustments. " + \
            "This task is a safety-critical task, requiring careful consideration of the situation. " + \
            "Front intruders are aircraft that are ahead of the ownship and closer to the next waypoint, no matter on the same route or a different route. " + \
            f"At each time step, the ownship can choose one of the following actions: 'Accelerate' (increase speed by {ownship_info['speed_change_per_second(m/s2)']} m/s per second), " + \
            f"'Decelerate' (decrease speed by -{ownship_info['speed_change_per_second(m/s2)']} m/s per second), or 'Hold' its current speed. " + \
            f"The intruders are ahead of the ownship, but there may be other intruders behind the ownship. " + \
            f"The ownship cannot unnecessarily decelerate since it might occlude the airspace for the intruders behind it." + \
            f"The simulation time step is {env.dt} seconds. \n" + \
            "Based on the following information about the ownship and its intruders, recommend one action to adjust " + \
            "its current speed to maintain safe separation. \nHere is the ownship information: \n"

    # prompt += f"Ownship ID: {ownship_info['agent_id']}, "
    prompt += f"Aircraft Type: {ownship_info['ac_type']}, "
    # prompt += f"Latitude: {ownship_info['ac_lat']}, Longitude: {ownship_info['ac_lon']}, "
    prompt += f"Next Waypoint ID: {ownship_info['next_wpt_id']}, Next Waypoint Type: {ownship_info['next_wpt_type']}, "
    prompt += f"Distance to Next Waypoint (m): {ownship_info['ac_dist_to_nxt_wpt(m)']}, "
    prompt += f"Speed (m/s): {ownship_info['speed(m/s)']}, "
    prompt += f"Speed Range (m/s): [{ownship_info['ac_min_spd(m/s)']}, {ownship_info['ac_max_spd(m/s)']}], "
    prompt += f"Speed Change per Second (m/s²): {ownship_info['speed_change_per_second(m/s2)']}, "
    # prompt += f"Acceleration (knot/s): {ownship_info['acceleration(knot/s)']}, "
    prompt += f"Heading (deg): {ownship_info['heading(deg)']} with heading scale {ownship_info['heading_scale']}, "
    # prompt += f"Altitude (ft): {ownship_info['altitude(ft)']}, "
    # prompt += f"Route ID: {ownship_info['ac_route_id']}, "
    prompt += f"Last Action: {ownship_info['ac_last_action']}, "
    
    safety_distance = env.get_agent_safety_distance(ownship_info['agent_id']) * 1852  # convert from nautical miles to meters
    ownship_collision_dist = env.get_agent_collision_distance(ownship_info['agent_id']) * 1852  # convert from nautical miles to meters
    # prompt += f"The safety distance to maintain from intruders is {safety_distance:.2f} (m), which is a safe boundary around the ownship. "
    prompt += f"The near mid-air collision distance is {ownship_collision_dist:.2f} (m), which is the most critical distance between the ownship and the intruder. \n"
    prompt += "Here are the intruder details: \n"
    prompt += f"Number of Intruders Ahead: {ownship_info['num_intruders_ahead']}. "
    if ownship_info['num_intruders_ahead'] == 1:
        prompt += "The first closest front intruder information: \n"
        # prompt += f"Intruder ID: {first_intruder_info['agent_id']}, "
        prompt += f"Type: {first_intruder_info['ac_type']}, "
        # prompt += f"Latitude: {first_intruder_info['ac_lat']}, Longitude: {first_intruder_info['ac_lon']}, "
        prompt += f"Next Waypoint ID: {first_intruder_info['next_wpt_id']}, Next Waypoint Type: {first_intruder_info['next_wpt_type']}, "
        prompt += f"Distance to Next Waypoint (m): {first_intruder_info['ac_dist_to_nxt_wpt(m)']}, "
        prompt += f"Speed (m/s): {first_intruder_info['speed(m/s)']}, "
        # prompt += f"Acceleration (knot/s): {first_intruder_info['acceleration(knot/s)']}, "
        prompt += f"Heading (deg): {first_intruder_info['heading(deg)']}, "
        # prompt += f"Altitude (ft): {first_intruder_info['altitude(m)']}, "
        # prompt += f"Route ID: {first_intruder_info['ac_route_id']}, "
        prompt += f"Distance to the ownship (m): {ownship_info['distance_to_intruder1(m)']}. "
        if ownship_info['intruder1_on_same_route']:
            prompt += "The first intruder and the ownship are on the same route. \n"
        else:
            prompt += "The first intruder and the ownship are not on the same route. \n"
        
    elif ownship_info['num_intruders_ahead'] == 2:
        prompt += "The first closest front intruder information: \n"
        # prompt += f"Intruder ID: {first_intruder_info['agent_id']}, "
        prompt += f"Type: {first_intruder_info['ac_type']}, "
        # prompt += f"Latitude: {first_intruder_info['ac_lat']}, Longitude: {first_intruder_info['ac_lon']}, "
        prompt += f"Next Waypoint ID: {first_intruder_info['next_wpt_id']}, Next Waypoint Type: {first_intruder_info['next_wpt_type']}, "
        prompt += f"Distance to Next Waypoint (m): {first_intruder_info['ac_dist_to_nxt_wpt(m)']}, "
        prompt += f"Speed (m/s): {first_intruder_info['speed(m/s)']}, "
        # prompt += f"Acceleration (knot/s): {first_intruder_info['acceleration(knot/s)']}, "
        prompt += f"Heading (deg): {first_intruder_info['heading(deg)']}, "
        # prompt += f"Altitude (ft): {first_intruder_info['altitude(m)']}, "
        # prompt += f"Route ID: {first_intruder_info['ac_route_id']}, "
        prompt += f"Distance to the ownship (m): {ownship_info['distance_to_intruder1(m)']}. "
        if ownship_info['intruder1_on_same_route']:
            prompt += "The first intruder and the ownship are on the same route. \n"
        else:
            prompt += "The first intruder and the ownship are not on the same route. \n"
        # prompt += f"Are the first intruder and the ownship on the same route? {ownship_info['intruder1_on_same_route']}. \n"
        
        prompt += "Tthe second closest front intruder information: \n"
        # prompt += f"Intruder ID: {second_intruder_info['agent_id']}, "
        prompt += f"Type: {second_intruder_info['ac_type']}, "
        # prompt += f"Latitude: {second_intruder_info['ac_lat']}, Longitude: {second_intruder_info['ac_lon']}, "
        prompt += f"Next Waypoint ID: {second_intruder_info['next_wpt_id']}, Next Waypoint Type: {second_intruder_info['next_wpt_type']}, "
        prompt += f"Distance to Next Waypoint (m): {second_intruder_info['ac_dist_to_nxt_wpt(m)']}, "
        prompt += f"Speed (m/s): {second_intruder_info['speed(m/s)']}, "
        # prompt += f"Acceleration (knot/s): {second_intruder_info['acceleration(knot/s)']}, "
        prompt += f"Heading (deg): {second_intruder_info['heading(deg)']}, "
        # prompt += f"Altitude (ft): {second_intruder_info['altitude(ft)']}, "
        # prompt += f"Route ID: {second_intruder_info['ac_route_id']}, "
        prompt += f"Distance to the ownship (m): {ownship_info['distance_to_intruder2(m)']}. "
        if ownship_info['intruder2_on_same_route']:
            prompt += "The second intruder and the ownship are on the same route. \n"
        else:
            prompt += "The second intruder and the ownship are not on the same route. \n"

    else:
        prompt += "There are no front intruders detected, so try to avoid occlusion. \n"

    prompt += f"Based on the above information, what actions should the ownship take? (Decelerate/Hold/Accelerate) \n"
    prompt_response = f"The recommended action for the ownship is to '{ownship_action}'. \n"
    return prompt, prompt_response

In [26]:
def get_ownship_prompt(env, ownship_intruder_info):
    """
    Returns a formatted string prompt for the ownship information.
    """
    ownship_info = ownship_intruder_info['ownship_info']
    first_intruder_info = ownship_intruder_info['first_closest_front_intruder_info']
    second_intruder_info = ownship_intruder_info['second_closest_front_intruder_info']
    ownship_action = ownship_intruder_info['ownship_action']

    prompt = "You are an airspace tactical deconfliction assistant. " + \
            "Your task is to help the ownship aircraft avoid collisions with front intruder aircraft by suggesting appropriate speed adjustments. " + \
            "This task is a safety-critical task, requiring careful consideration of the situation. " + \
            "Front intruders are aircraft that are ahead of the ownship and closer to the next waypoint, no matter on the same route or a different route. " + \
            f"At each time step, the ownship can choose one of the following actions: 'Accelerate' (increase speed by {ownship_info['speed_change_per_second(m/s2)']} m/s per second), " + \
            f"'Decelerate' (decrease speed by -{ownship_info['speed_change_per_second(m/s2)']} m/s per second), or 'Hold' its current speed. " + \
            f"The intruders are ahead of the ownship, but there may be other intruders behind the ownship. " + \
            f"The ownship cannot unnecessarily decelerate since it might occlude the airspace for the intruders behind it." + \
            f"The simulation time step is {env.dt} seconds. \n" + \
            "Based on the following information about the ownship and its intruders, recommend one action to adjust " + \
            "its current speed to maintain safe separation. \nHere is the ownship information: \n"

    # prompt += f"Ownship ID: {ownship_info['agent_id']}, "
    # prompt += f"Aircraft Type: {ownship_info['ac_type']}, "
    # prompt += f"Latitude: {ownship_info['ac_lat']}, Longitude: {ownship_info['ac_lon']}, "
    prompt += f"Speed (m/s): {ownship_info['speed(m/s)']}, "
    prompt += f"Speed Range (m/s): [{ownship_info['ac_min_spd(m/s)']}, {ownship_info['ac_max_spd(m/s)']}], "
    prompt += f"Speed Change per Second (m/s²): {ownship_info['speed_change_per_second(m/s2)']}, "
    # prompt += f"Next Waypoint ID: {ownship_info['next_wpt_id']}, Next Waypoint Type: {ownship_info['next_wpt_type']}, "
    # prompt += f"Distance to Next Waypoint (m): {ownship_info['ac_dist_to_nxt_wpt(m)']}, "

    # prompt += f"Acceleration (knot/s): {ownship_info['acceleration(knot/s)']}, "
    # prompt += f"Heading (deg): {ownship_info['heading(deg)']} with heading scale {ownship_info['heading_scale']}, "
    # prompt += f"Altitude (ft): {ownship_info['altitude(ft)']}, "
    # prompt += f"Route ID: {ownship_info['ac_route_id']}, "
    # prompt += f"Last Action: {ownship_info['ac_last_action']}, "

    # safety_distance = int(env.get_agent_safety_distance(ownship_info['agent_id']) * 1852)  # convert from nautical miles to meters
    # ownship_collision_dist = int(env.get_agent_collision_distance(ownship_info['agent_id']) * 1852)  # convert from nautical miles to meters
    # prompt += f"The safety distance to maintain from intruders is {safety_distance:.2f} (m), which is a safe boundary around the ownship. "
    # prompt += f"The near mid-air collision distance is {ownship_collision_dist:.2f} (m), which is the most critical distance between the ownship and the intruder. \n"
    prompt += "Here are the intruder details: \n"
    prompt += f"There are {ownship_info['num_intruders_ahead']} intruders ahead. "
    if ownship_info['num_intruders_ahead'] == 1:
        prompt += "The first closest front intruder information: \n"
        # prompt += f"Intruder ID: {first_intruder_info['agent_id']}, "
        # prompt += f"Type: {first_intruder_info['ac_type']}, "
        # prompt += f"Latitude: {first_intruder_info['ac_lat']}, Longitude: {first_intruder_info['ac_lon']}, "
        # prompt += f"Next Waypoint ID: {first_intruder_info['next_wpt_id']}, Next Waypoint Type: {first_intruder_info['next_wpt_type']}, "
        prompt += f"Distance to Next Waypoint (m): {first_intruder_info['ac_dist_to_nxt_wpt(m)']}, "
        prompt += f"Speed (m/s): {first_intruder_info['speed(m/s)']}, "
        # prompt += f"Acceleration (knot/s): {first_intruder_info['acceleration(knot/s)']}, "
        # prompt += f"Heading (deg): {first_intruder_info['heading(deg)']}, "
        # prompt += f"Altitude (ft): {first_intruder_info['altitude(m)']}, "
        # prompt += f"Route ID: {first_intruder_info['ac_route_id']}, "
        # prompt += f"Distance to the ownship (m): {ownship_info['distance_to_intruder1(m)']}. "
        if ownship_info['intruder1_on_same_route']:
            prompt += "The first intruder and the ownship are on the same route. \n"
        else:
            prompt += "The first intruder and the ownship are not on the same route. \n"

        # format time to collision to 2 decimal places
        prompt += f"Ownship's time to collision with the first intruder is {ownship_info['time_to_collision_with_inruder1(s)']:.2f} (s). "

    elif ownship_info['num_intruders_ahead'] == 2:
        prompt += "The first closest front intruder information: \n"
        # prompt += f"Intruder ID: {first_intruder_info['agent_id']}, "
        # prompt += f"Type: {first_intruder_info['ac_type']}, "
        # prompt += f"Latitude: {first_intruder_info['ac_lat']}, Longitude: {first_intruder_info['ac_lon']}, "
        # prompt += f"Next Waypoint ID: {first_intruder_info['next_wpt_id']}, Next Waypoint Type: {first_intruder_info['next_wpt_type']}, "
        prompt += f"Distance to Next Waypoint (m): {first_intruder_info['ac_dist_to_nxt_wpt(m)']}, "
        prompt += f"Speed (m/s): {first_intruder_info['speed(m/s)']}, "
        # prompt += f"Acceleration (knot/s): {first_intruder_info['acceleration(knot/s)']}, "
        # prompt += f"Heading (deg): {first_intruder_info['heading(deg)']}, "
        # prompt += f"Altitude (ft): {first_intruder_info['altitude(m)']}, "
        # prompt += f"Route ID: {first_intruder_info['ac_route_id']}, "
        # prompt += f"Distance to the ownship (m): {ownship_info['distance_to_intruder1(m)']}. "
        if ownship_info['intruder1_on_same_route']:
            prompt += "The first intruder and the ownship are on the same route. \n"
        else:
            prompt += "The first intruder and the ownship are not on the same route. \n"

        prompt += f"Ownship's time to collision with the first intruder is {ownship_info['time_to_collision_with_inruder1(s)']:.2f} (s). "

        prompt += "The second closest front intruder information: \n"
        # prompt += f"Intruder ID: {second_intruder_info['agent_id']}, "
        # prompt += f"Type: {second_intruder_info['ac_type']}, "
        # prompt += f"Latitude: {second_intruder_info['ac_lat']}, Longitude: {second_intruder_info['ac_lon']}, "
        # prompt += f"Next Waypoint ID: {second_intruder_info['next_wpt_id']}, Next Waypoint Type: {second_intruder_info['next_wpt_type']}, "
        prompt += f"Distance to Next Waypoint (m): {second_intruder_info['ac_dist_to_nxt_wpt(m)']}, "
        prompt += f"Speed (m/s): {second_intruder_info['speed(m/s)']}, "
        # prompt += f"Acceleration (knot/s): {second_intruder_info['acceleration(knot/s)']}, "
        # prompt += f"Heading (deg): {second_intruder_info['heading(deg)']}, "
        # prompt += f"Altitude (ft): {second_intruder_info['altitude(ft)']}, "
        # prompt += f"Route ID: {second_intruder_info['ac_route_id']}, "
        # prompt += f"Distance to the ownship (m): {ownship_info['distance_to_intruder2(m)']}. "
        if ownship_info['intruder2_on_same_route']:
            prompt += "The second intruder and the ownship are on the same route. \n"
        else:
            prompt += "The second intruder and the ownship are not on the same route. \n"

        prompt += f"Ownship's time to collision with the second intruder is {ownship_info['time_to_collision_with_inruder2(s)']:.2f} (s). "

    else:
        prompt += "There are no front intruders detected, so try to avoid occlusion. \n"

    prompt += f"Based on the above information, what actions should the ownship take? (Decelerate/Hold/Accelerate) \n"

    prompt_response = f"The recommended action for the ownship is to '{ownship_action}'. \n"
    
    return prompt, prompt_response

In [3]:
def get_ownship_prompt(env, ownship_intruder_info):
    """
    Returns a formatted string prompt for the ownship information.
    """
    ownship_info = ownship_intruder_info['ownship_info']
    first_intruder_info = ownship_intruder_info['first_closest_front_intruder_info']
    second_intruder_info = ownship_intruder_info['second_closest_front_intruder_info']
    ownship_action = ownship_intruder_info['ownship_action']

    system_prompt = """You are an airspace tactical deconfliction assistant. At each time step, an ownship agent is approaching a bottleneck waypoint, 
    such as merging or intersections points, where other agents (intruders) are approaching as well. Based on the information of the intruders, 
    the ownship should take an action to avoid collisions. The ownship agent only has access to the information of the front intruders, 
    but there might be other intruders behind the ownship. Your task is to help the ownship aircraft avoid collisions with front intruder aircraft by 
    suggesting appropriate speed adjustments. The ownship cannot unnecessarily decelerate since it might occlude the airspace for other agents behind it.
    Now, given the ownship state: [v, v_min, v_max delta_d1, delta_d2, delta_dwpt1, delta_dwpt2, delta_v1, delta_v2, same_route1, same_route2],
    v: ownship speed(m/s), 
    v_min: ownship minimum speed(m/s), 
    v_max: ownship maximum speed(m/s),
    d1: the absolute distance between the first front intruder (if any) and ownship(m), 
    d2: the absolute distance between the second front intruder (if any) and ownship(m), 
    delta_dwpt1: the absolute difference between the distance to the next waypoint between the first front intruder (if any) and ownship(m), 
    delta_dwpt2: the absolute difference between the distance to the next waypoint between the second front intruder (if any) and ownship(m), 
    delta_v1: signed speed difference between the first front intruder and ownship(m/s), 
    delta_v2: signed speed difference between the second front intruder and ownship(m/s), 
    same_route1: whether the first front intruder is on the same route as the ownship (1: yes, 0: no), 
    same_route2: whether the second front intruder is on the same route as the ownship (1: yes, 0: no). 
    If one of the values is None, it means there is no such front intruder.
    Based on the above information, decide the ownship action: Decelerate / Hold / Accelerate. 
    Never include reasoning, explanations, or any text other than the single chosen action.
    """

    user_prompt = """Here is the ownship state information:
    [v, v_min, v_max, d1, d2, delta_dwpt1, delta_dwpt2, delta_v1, delta_v2, same_route1, same_route2] = [{v:.2f}, {v_min:.2f}, {v_max:.2f}, {d1:.2f}, {d2:.2f}, {delta_dwpt1:.2f}, {delta_dwpt2:.2f}, {delta_v1:.2f}, {delta_v2:.2f}, {same_route1:.0f}, {same_route2:.0f}]. 
    Based on the above information, what actions should the ownship take? Your output must contain exactly one word and no punctuation."""
    
    ownship_spd = ownship_info['speed(m/s)']
    ownship_min_spd = ownship_info['ac_min_spd(m/s)']
    ownship_max_spd = ownship_info['ac_max_spd(m/s)']
    d1 = ownship_info['distance_to_intruder1(m)'] if ownship_info['num_intruders_ahead'] >= 1 else None
    d2 = ownship_info['distance_to_intruder2(m)'] if ownship_info['num_intruders_ahead'] >= 2 else None
    delta_dwpt1 = abs(first_intruder_info['ac_dist_to_nxt_wpt(m)'] - ownship_info['ac_dist_to_nxt_wpt(m)']) if ownship_info['num_intruders_ahead'] >= 1 else None
    delta_dwpt2 = abs(second_intruder_info['ac_dist_to_nxt_wpt(m)'] - ownship_info['ac_dist_to_nxt_wpt(m)']) if ownship_info['num_intruders_ahead'] >= 2 else None
    delta_v1 = first_intruder_info['speed(m/s)'] - ownship_spd if ownship_info['num_intruders_ahead'] >= 1 else None
    delta_v2 = second_intruder_info['speed(m/s)'] - ownship_spd if ownship_info['num_intruders_ahead'] >= 2 else None
    same_route1 = ownship_info['intruder1_on_same_route'] if ownship_info['num_intruders_ahead'] >= 1 else None
    same_route2 = ownship_info['intruder2_on_same_route'] if ownship_info['num_intruders_ahead'] >= 2 else None

    user_prompt = user_prompt.format(
        v=ownship_spd,
        v_min=ownship_min_spd,
        v_max=ownship_max_spd,
        d1=d1 if d1 is not None else float('nan'),
        d2=d2 if d2 is not None else float('nan'),
        delta_dwpt1=delta_dwpt1 if delta_dwpt1 is not None else float('nan'),
        delta_dwpt2=delta_dwpt2 if delta_dwpt2 is not None else float('nan'),
        delta_v1=delta_v1 if delta_v1 is not None else float('nan'),
        delta_v2=delta_v2 if delta_v2 is not None else float('nan'),
        same_route1=same_route1 if same_route1 is not None else float('nan'),
        same_route2=same_route2 if same_route2 is not None else float('nan')
    )

    prompt_response = f"The recommended action for the ownship is to '{ownship_action}'. \n"
    
    return system_prompt, user_prompt, prompt_response

In [3]:
import json
import pandas as pd

# Load the JSONL file
data = []
with open('raw_ownship_intruders_data_Oct30.jsonl', 'r') as file:
    for line in file:
        data.append(json.loads(line.strip()))

print(f"Loaded {len(data)} rows from the file")

Loaded 38014 rows from the file


In [4]:
for row in data[0:2]:
    print(row)

{'ownship_info': {'agent_id': 'AB0001', 'ac_type': 'Google X-Wing', 'ac_lat': 33.175611, 'ac_lon': -96.822764, 'next_wpt_id': 'WP3', 'next_wpt_type': 'Regular', 'ac_dist_to_nxt_wpt(m)': 3513.18, 'speed(m/s)': 15.95, 'ac_min_spd(m/s)': 0.0, 'ac_max_spd(m/s)': 30.87, 'speed_change_per_second(m/s2)': 1.03, 'heading(deg)': 212.2, 'heading_scale': '{North=0, East=90, South=180, West=270}', 'altitude(m)': 1389.36, 'ac_route_id': 'R_2', 'ac_last_action': 'accelerate', 'num_intruders_ahead': 0, 'ac_desired_spd(m/s)': 23.15, 'intruder1_on_same_route': None, 'intruder2_on_same_route': None, 'distance_to_intruder1(m)': None, 'distance_to_intruder2(m)': None, 'time_to_collision_with_inruder1(s)': None, 'time_to_collision_with_inruder2(s)': None, 'did_ownship_have_NMAC': False}, 'first_closest_front_intruder_info': [], 'second_closest_front_intruder_info': [], 'ownship_action': 'Accelerate'}
{'ownship_info': {'agent_id': 'AE0001', 'ac_type': 'Google X-Wing', 'ac_lat': 33.136842, 'ac_lon': -96.80925

In [11]:
for row in data[:1]:  # Display first 5 rows
    system_prompt, user_prompt, prompt_response = get_ownship_prompt(env, row)
    print(system_prompt)
    print(user_prompt)
    print(prompt_response)


    You are an airspace tactical deconfliction assistant. At each time step, an ownship agent is approaching a bottleneck waypoint, 
    such as merging or intersections points, where other agents (intruders) are approaching as well. Based on the information of the intruders, 
    the ownship should take an action to avoid collisions. The ownship agent only has access to the information of the front intruders, 
    but there might be other intruders behind the ownship. Your task is to help the ownship aircraft avoid collisions with front intruder aircraft by 
    suggesting appropriate speed adjustments. The ownship cannot unnecessarily decelerate since it might occlude the airspace for other agents behind it.
    Now, given the ownship state: [v, v_min, v_max delta_d1, delta_d2, delta_dwpt1, delta_dwpt2, delta_v1, delta_v2, same_route1, same_route2],
    v: ownship speed(m/s), 
    v_min: ownship minimum speed(m/s), 
    v_max: ownship maximum speed(m/s),
    d1: the absolute distanc

In [20]:
with open('ownship_prompts_data.jsonl', 'w') as outfile:
    for row in data:  
        prompt = {}
        system_prompt, user_prompt, prompt_response = get_ownship_prompt(env, row)
        prompt['system_prompt'] = system_prompt
        prompt['user_prompt'] = user_prompt
        prompt['prompt_response'] = prompt_response
        outfile.write(json.dumps(prompt) + '\n')